# EDA Notebook\n\nUse this notebook for exploration.

# 1. Imports

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

print("✅ Libraries loaded")

📌 CELL 2 — Load all datasets

In [ ]:
customers = pd.read_csv("../inputs/customers.csv", parse_dates=["signup_date"])
transactions = pd.read_csv("../inputs/transactions.csv", parse_dates=["transaction_date"])
events = pd.read_csv("../inputs/events.csv", parse_dates=["event_date"])
support = pd.read_csv("../inputs/support.csv", parse_dates=["ticket_date"])

customers.head()
transactions.head()
events.head()
support.head()

📌 CELL 3 — Dataset Shapes

In [ ]:
print("Customers:", customers.shape)
print("Transactions:", transactions.shape)
print("Events:", events.shape)
print("Support:", support.shape)


📌 CELL 4 — Missing Values Report

In [ ]:
print("Missing values in Customers:\n", customers.isnull().sum(), "\n")
print("Missing values in Transactions:\n", transactions.isnull().sum(), "\n")
print("Missing values in Events:\n", events.isnull().sum(), "\n")
print("Missing values in Support:\n", support.isnull().sum(), "\n")


📌 CELL 5 — Basic Statistics

In [ ]:
customers.describe(include="all")
transactions.describe(include="all")
events.describe(include="all")
support.describe(include="all")

📌 CELL 6 — Visualizing Customer Attributes

In [ ]:
# 🔸 Plan Distribution
sns.countplot(data=customers, x="plan")
plt.title("Customer Plan Distribution")
plt.show()
# 🔸 Monthly Transactions Over Time
# 🔸 Churn Distribution
sns.countplot(data=customers, x="churn")
plt.title("Churn Distribution")
plt.show()
# 🔸 Monthly Fee Distribution
sns.histplot(customers["monthly_fee"], kde=True, bins=30)
plt.title("Monthly Fee Distribution")
plt.show()


📌 CELL 7 — Transaction Behavior

In [ ]:
# Total Spending per Customer
trans_agg = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    trans_count=("amount", "count"),
    last_transaction=("transaction_date", "max")
).reset_index()

trans_agg.head()


📌 CELL 8 — Event Behavior (Login Activity)

In [ ]:
events["event_type"].value_counts()
# Logins in last 7 / 30 / 90 days
ref_date = customers["signup_date"].max() + pd.Timedelta(days=365)

events["days_from_ref"] = (ref_date - events["event_date"]).dt.days

login_7 = events[events["days_from_ref"] <= 7].groupby("customer_id").size().rename("logins_7d")
login_30 = events[events["days_from_ref"] <= 30].groupby("customer_id").size().rename("logins_30d")
login_90 = events[events["days_from_ref"] <= 90].groupby("customer_id").size().rename("logins_90d")

logins = pd.concat([login_7, login_30, login_90], axis=1).fillna(0).reset_index()
logins.head()


 📌 CELL 9 — Support Metrics

In [ ]:
support_agg = support.groupby("customer_id").agg(
    tickets=("issue_type", "count"),
    avg_resolution=("resolution_days", "mean")
).reset_index()

support_agg.head()


📌 CELL 10 — Merge All Features

In [ ]:
df = customers.merge(trans_agg, on="customer_id", how="left")
df = df.merge(logins, on="customer_id", how="left")
df = df.merge(support_agg, on="customer_id", how="left")

df = df.fillna(0)
df.head()


📌 CELL 11 — Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.corr(), annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()
# Spending vs. Churn
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="churn", y="total_spend")
plt.title("Total Spending vs. Churn")
plt.show()

# 2. Load the 4 input files


In [13]:
import os
from pathlib import Path

def safe_read_csv(path, parse_dates=None):
	"""Read CSV if it exists, otherwise return an empty DataFrame and log a warning."""
	if os.path.exists(path):
		try:
			return pd.read_csv(path, parse_dates=parse_dates)
		except Exception as e:
			print(f"❌ Error reading {path}: {e}")
			return pd.DataFrame()
	else:
		print(f"⚠️ File not found: {path}. Creating empty DataFrame.")
		return pd.DataFrame()

base = Path(r"F:\finalYearProject\project_code\inputs").expanduser().resolve()

customers = safe_read_csv(str(base / "customers.csv"), parse_dates=["signup_date"])
transactions = safe_read_csv(str(base / "transactions.csv"), parse_dates=["transaction_date"])
events = safe_read_csv(str(base / "events.csv"), parse_dates=["event_date"])
support = safe_read_csv(str(base / "support.csv"), parse_dates=["ticket_date"])

print("Resolved input directory:", base)

print("✅ Data loaded (missing files result in empty DataFrames)\n")

print("Customers (top 5):")
print(customers.head(), "\n")

print("Transactions (top 5):")
print(transactions.head(), "\n")

print("Events (top 5):")
print(events.head(), "\n")

print("Support  (top 5):")
print(support.head(), "\n")

# 3. Shapes of each dataset


In [14]:
print("📏 SHAPES")
print("Customers:", customers.shape)
print("Transactions:", transactions.shape)
print("Events:", events.shape)
print("Support:", support.shape, "\n")


# 4. Missing values


In [15]:
print("❓ MISSING VALUES\n")

def missing_report(df, name):
    print(f"--- {name} ---")
    print(df.isnull().sum())
    print()

missing_report(customers, "customers")
missing_report(transactions, "transactions")
missing_report(events, "events")
missing_report(support, "support")

# 5. Basic statistics on numerical columns (customers)


In [16]:
print("📊 Customers describe():")
print(customers.describe(include="all"))

# 6. Simple EDA plots (run only if in notebook / interactive)


In [17]:
def quick_plots():
    # Plan distribution if 'plan' column exists
    if "plan" in customers.columns:
        customers["plan"].value_counts().plot(kind="bar")
        plt.title("Plan Distribution")
        plt.xlabel("Plan")
        plt.ylabel("Count")
        plt.show()

    # Churn distribution if 'churn' column exists
    if "churn" in customers.columns:
        customers["churn"].value_counts().plot(kind="bar")
        plt.title("Churn Distribution")
        plt.xlabel("Churn (0 = no, 1 = yes)")
        plt.ylabel("Count")
        plt.show()

    # Transaction amount distribution
    if "amount" in transactions.columns:
        transactions["amount"].hist(bins=40)
        plt.title("Transaction Amount Distribution")
        plt.xlabel("Amount")
        plt.ylabel("Frequency")
        plt.show()

# Uncomment this if running in Jupyter or want to see plots:
quick_plots()

# 7. Simple feature engineering preview:
  - Total amount per customer
  - Transaction count per customer
  - Last transaction date

In [18]:
print("\n🧮 Feature engineering preview...")

trans_agg = (
    transactions
    .groupby("customer_id")
    .agg(
        total_amount=("amount", "sum"),
        trans_count=("amount", "count"),
        last_trans_date=("transaction_date", "max"),
    )
    .reset_index()
)

print("Transaction aggregates (top 5):")
print(trans_agg.head(), "\n")

# Merge with customers as an example
customers_fe = customers.merge(trans_agg, on="customer_id", how="left")

print("Customers + basic features (top 5):")
print(customers_fe.head(), "\n")


# 8. Simple model example: churn prediction using few features
    -(for proper training, use train.py later)

In [20]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

print("🤖 Simple churn model demo...")

df = customers_fe.copy()


# Drop rows where churn is missing (if any)


In [22]:
df = df[~df["churn"].isna()]
df


# Encode plan if exists


In [23]:
if "plan" in df.columns:
    le_plan = LabelEncoder()
    df["plan_enc"] = le_plan.fit_transform(df["plan"])
else:
    df["plan_enc"] = 0  # fallback

# Fill NaNs from feature engineering


In [26]:
for col in ["total_amount", "trans_count"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)
        


# Select a small set of features


In [27]:
feature_cols = []
for col in ["monthly_fee", "recency_days", "total_amount", "trans_count", "plan_enc"]:
    if col in df.columns:
        feature_cols.append(col)

X = df[feature_cols]
y = df["churn"].astype(int)

print("Using features:", feature_cols)
print("X shape:", X.shape, "y shape:", y.shape, "\n")

# Train-test split



In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("✅ Classification report:")
print(classification_report(y_test, y_pred))

try:
    auc = roc_auc_score(y_test, y_proba)
    print("ROC-AUC:", round(auc, 4))
except Exception:
    pass

print("\n✅ Notebook-style analysis done.")